# Advanced Models — RandomForest, XGBoost, and Deep Neural Network

**Goal:** Compare LogisticRegression (our baseline winner from notebook 5) against three more powerful models:
1. **RandomForest** — ensemble of decision trees, captures non-linear interactions
2. **XGBoost** — gradient-boosted trees, typically the best tabular data model
3. **Deep Neural Network** — multi-layer network with residual connections

For each model we report: accuracy, MSE, R², F1, parameter count, and PnL simulation.

**Training data:** Filtered to the optimal window from notebook 5: `elapsed >= 30%`, `winner_bid <= 0.85`.

**Why these models?**
- LogisticRegression finds linear boundaries. If the signal is non-linear (e.g. "UP when BTC velocity is moderate but not extreme"), it can't capture it.
- RandomForest and XGBoost handle non-linear feature interactions natively via tree splits.
- DNN can learn arbitrary functions if given enough data, but risks overfitting on small datasets.

In [ ]:
import json
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import xgboost as xgb
from evaluator import Evaluator
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

MAX_BID = 0.85
device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## 1. Load and prepare data

**What:** Same as notebook 5 — load features, filter to the optimal trading window, split 80/20.

In [ ]:
DATA_PATH = Path("../data/legacy_features.jsonl")
OPTIMAL_ELAPSED = 0.30

rows = []
with open(DATA_PATH) as f:
    for line in f:
        rows.append(json.loads(line))

df = pd.DataFrame(rows)
df["target"] = (df["outcome"] == "UP").astype(int)
df["winner_bid"] = df[["up_best_bid", "down_best_bid"]].max(axis=1)

NON_FEATURE_COLS = {
    "candle_id",
    "timestamp",
    "elapsed_pct",
    "btc_price",
    "up_best_bid",
    "up_best_ask",
    "up_bid_depth",
    "up_ask_depth",
    "down_best_bid",
    "down_best_ask",
    "down_bid_depth",
    "down_ask_depth",
    "market_volume",
    "outcome",
    "target",
    "winner_bid",
}
feature_cols = [c for c in df.columns if c not in NON_FEATURE_COLS]
df[feature_cols] = df[feature_cols].fillna(0.0)

# Filter to optimal window
mask = (df["elapsed_pct"] >= OPTIMAL_ELAPSED) & (df["winner_bid"] <= MAX_BID)
df_f = df[mask].copy()

scaler = StandardScaler()
X = scaler.fit_transform(df_f[feature_cols].values)
y = df_f["target"].values
asks = df_f["up_best_ask"].values
candle_ids = df_f["candle_id"].values

# Candle-level split — no snapshot leakage
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=candle_ids))
X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]
asks_train, asks_test = asks[train_idx], asks[test_idx]

train_candles = set(candle_ids[train_idx])
test_candles = set(candle_ids[test_idx])
print(
    f"Features: {len(feature_cols)}, Train: {X_train.shape[0]:,} ({len(train_candles)} candles), Test: {X_test.shape[0]:,} ({len(test_candles)} candles)"
)
print(f"Candle overlap: {len(train_candles & test_candles)} (should be 0)")
print(f"UP rate: train={y_train.mean() * 100:.1f}% test={y_test.mean() * 100:.1f}%")

## 2. Helper functions

**What:** PnL simulation and parameter counting — reused across all models.

In [ ]:
def simulate_pnl(pred, y_te, asks_te):
    """PnL for UP-only betting."""
    pnl, nb, wins = 0.0, 0, 0
    for p, t, a in zip(pred, y_te, asks_te, strict=True):
        if p == 1 and np.isfinite(a) and 0 < a < MAX_BID:
            nb += 1
            if t == 1:
                pnl += 1.0 - a
                wins += 1
            else:
                pnl -= a
    wr = wins / nb if nb > 0 else 0
    valid_asks = [a for p, a in zip(pred, asks_te, strict=True) if p == 1 and np.isfinite(a) and 0 < a < MAX_BID]
    avg_a = np.mean(valid_asks) if valid_asks else 0
    roi = pnl / (nb * avg_a) * 100 if nb > 0 and avg_a > 0 else 0
    return {"pnl": pnl, "n_bets": nb, "wins": wins, "win_rate": wr, "avg_entry": avg_a, "roi": roi}


def count_params_sklearn(model):
    """Estimate parameter count for sklearn models."""
    if hasattr(model, "estimators_"):
        # RandomForest / ensemble: sum of all node counts across all trees
        total = sum(est.tree_.node_count for est in model.estimators_)
        return total * 2  # each node has a feature index + threshold
    if hasattr(model, "coef_"):
        return model.coef_.size + model.intercept_.size
    return 0


def count_params_torch(model):
    """Count trainable parameters in a PyTorch model."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


all_results = []  # collect results for final comparison

## 3. Model 1 — LogisticRegression (baseline)

**What:** Our winner from notebook 5. Re-train here so all models use the exact same train/test split.

**Parameters:** `n_features + 1` (one weight per feature + intercept). With 54 features → 55 parameters. The simplest model.

In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)

y_prob_lr = lr.predict_proba(X_test)[:, 1]
y_pred_lr = lr.predict(X_test)
pnl_lr = simulate_pnl(y_pred_lr, y_test, asks_test)
n_params_lr = count_params_sklearn(lr)

ev_lr = Evaluator(y_test, y_pred_lr, y_prob_lr, title="LogisticRegression")
ev_lr.full_report()
print(f"\nParameters: {n_params_lr:,}")
print(
    f"PnL: ${pnl_lr['pnl']:+.2f}  Bets: {pnl_lr['n_bets']}  WR: {pnl_lr['win_rate'] * 100:.1f}%  ROI: {pnl_lr['roi']:+.1f}%"
)

all_results.append({"model": "LogisticRegression", "params": n_params_lr, "ev": ev_lr, "pnl": pnl_lr})

## 4. Model 2 — RandomForest

**What:** An ensemble of 200 decision trees. Each tree sees a random subset of features and samples. The final prediction is the majority vote.

**Why it might beat LogisticRegression:** Trees can split on "if btc_velocity > 0.3 AND imbalance > 1.5" — non-linear interactions that a linear model can't capture.

**Parameters:** Each tree has many nodes, each with a feature split and threshold. Total parameter count = sum of nodes across all trees × 2.

In [ ]:
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_leaf=20,
    random_state=42,
    n_jobs=-1,
)
rf.fit(X_train, y_train)

y_prob_rf = rf.predict_proba(X_test)[:, 1]
y_pred_rf = rf.predict(X_test)
pnl_rf = simulate_pnl(y_pred_rf, y_test, asks_test)
n_params_rf = count_params_sklearn(rf)

ev_rf = Evaluator(y_test, y_pred_rf, y_prob_rf, title="RandomForest")
ev_rf.full_report()
print(f"\nParameters: {n_params_rf:,}")
print(
    f"PnL: ${pnl_rf['pnl']:+.2f}  Bets: {pnl_rf['n_bets']}  WR: {pnl_rf['win_rate'] * 100:.1f}%  ROI: {pnl_rf['roi']:+.1f}%"
)

all_results.append({"model": "RandomForest", "params": n_params_rf, "ev": ev_rf, "pnl": pnl_rf})

## 5. Model 3 — XGBoost

**What:** Gradient-boosted decision trees. Unlike RandomForest (parallel trees), XGBoost builds trees *sequentially* — each new tree corrects errors from previous trees. Typically the strongest model for tabular data.

**Hyperparameters:**
- `n_estimators=500` — 500 sequential trees
- `learning_rate=0.05` — conservative learning rate (slower but more precise)
- `max_depth=6` — shallow trees (reduces overfitting, each tree captures a simple pattern)
- `eval_metric='logloss'` — optimizes log-loss (same as LogisticRegression)
- `early_stopping_rounds=20` — stops if validation loss doesn't improve for 20 rounds

In [ ]:
xgb_model = xgb.XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    min_child_weight=10,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="logloss",
    early_stopping_rounds=20,
    n_jobs=-1,
)

# Split a validation set from training for early stopping
X_tr_xgb, X_val_xgb, y_tr_xgb, y_val_xgb = train_test_split(
    X_train, y_train, test_size=0.1, random_state=42, stratify=y_train
)
xgb_model.fit(X_tr_xgb, y_tr_xgb, eval_set=[(X_val_xgb, y_val_xgb)], verbose=False)
print(f"XGBoost stopped at {xgb_model.best_iteration} trees (of {xgb_model.n_estimators} max)")

y_prob_xgb = xgb_model.predict_proba(X_test)[:, 1]
y_pred_xgb = xgb_model.predict(X_test)
pnl_xgb = simulate_pnl(y_pred_xgb, y_test, asks_test)
n_params_xgb = sum(t.count("leaf") + t.count("yes") for t in xgb_model.get_booster().get_dump())

ev_xgb = Evaluator(y_test, y_pred_xgb, y_prob_xgb, title="XGBoost")
ev_xgb.full_report()
print(f"\nParameters: ~{n_params_xgb:,} (nodes across all trees)")
print(
    f"PnL: ${pnl_xgb['pnl']:+.2f}  Bets: {pnl_xgb['n_bets']}  WR: {pnl_xgb['win_rate'] * 100:.1f}%  ROI: {pnl_xgb['roi']:+.1f}%"
)

all_results.append({"model": "XGBoost", "params": n_params_xgb, "ev": ev_xgb, "pnl": pnl_xgb})

## 6. Model 4 — Deep Neural Network

**What:** A multi-layer neural network with residual connections (skip connections), trained with PyTorch.

**Architecture:**
```
Input (54 features)
  → Linear(128) → LayerNorm → ReLU → Dropout(0.3)
  → ResidualBlock(128) × 4
  → Linear(1) → Sigmoid
```

Each ResidualBlock: `Linear(128→128) → LayerNorm → ReLU → Dropout → Linear(128→128) → LayerNorm + skip connection → ReLU`

**Why residual connections?** They let gradients flow directly through the network, making deep architectures trainable. Without them, a 6-layer network would struggle to learn.

**Training:** Binary cross-entropy loss (same objective as LogisticRegression), AdamW optimizer, cosine annealing LR schedule, gradient clipping.

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, hidden_size, dropout_prob=0.3):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.LayerNorm(hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout_prob),
            nn.Linear(hidden_size, hidden_size),
            nn.LayerNorm(hidden_size),
        )
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.relu(x + self.block(x))


class BetPredictor(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_blocks=4, dropout_prob=0.3):
        super().__init__()
        self.input_layer = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.LayerNorm(hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout_prob),
        )
        self.blocks = nn.Sequential(*[ResidualBlock(hidden_size, dropout_prob) for _ in range(num_blocks)])
        self.output_layer = nn.Sequential(
            nn.Linear(hidden_size, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        x = self.input_layer(x)
        x = self.blocks(x)
        return self.output_layer(x)


n_features = X_train.shape[1]
dnn = BetPredictor(input_size=n_features).to(device)
n_params_dnn = count_params_torch(dnn)
print(f"DNN architecture: {n_features} → 128 → 4×ResBlock(128) → 1")
print(f"Trainable parameters: {n_params_dnn:,}")
print(dnn)

In [ ]:
# Prepare PyTorch datasets
X_tr_t = torch.FloatTensor(X_train).to(device)
y_tr_t = torch.FloatTensor(y_train).unsqueeze(1).to(device)
X_te_t = torch.FloatTensor(X_test).to(device)

train_ds = TensorDataset(X_tr_t, y_tr_t)
train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)

# Training setup
criterion = nn.BCELoss()
optimizer = torch.optim.AdamW(dnn.parameters(), lr=0.001, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20, eta_min=1e-5)

EPOCHS = 30
best_val_loss = float("inf")
best_state = None
patience = 5
no_improve = 0

# Split validation from training for early stopping
X_tr_dnn, X_val_dnn, y_tr_dnn, y_val_dnn = train_test_split(
    X_train, y_train, test_size=0.1, random_state=42, stratify=y_train
)
X_val_t = torch.FloatTensor(X_val_dnn).to(device)
y_val_t = torch.FloatTensor(y_val_dnn).unsqueeze(1).to(device)

# Rebuild loader with actual training subset
X_tr_dnn_t = torch.FloatTensor(X_tr_dnn).to(device)
y_tr_dnn_t = torch.FloatTensor(y_tr_dnn).unsqueeze(1).to(device)
train_ds = TensorDataset(X_tr_dnn_t, y_tr_dnn_t)
train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)

print(f"Training DNN for up to {EPOCHS} epochs (early stopping patience={patience})")
for epoch in range(EPOCHS):
    dnn.train()
    train_loss = 0.0
    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()
        out = dnn(batch_x)
        loss = criterion(out, batch_y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(dnn.parameters(), max_norm=1.0)
        optimizer.step()
        train_loss += loss.item() * len(batch_x)
    train_loss /= len(train_ds)

    # Validation
    dnn.eval()
    with torch.no_grad():
        val_out = dnn(X_val_t)
        val_loss = criterion(val_out, y_val_t).item()
        val_acc = ((val_out >= 0.5).float() == y_val_t).float().mean().item()

    scheduler.step()
    lr_now = scheduler.get_last_lr()[0]

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = {k: v.clone() for k, v in dnn.state_dict().items()}
        no_improve = 0
        marker = " *"
    else:
        no_improve += 1
        marker = ""

    if (epoch + 1) % 5 == 0 or epoch < 3 or marker:
        print(
            f"  Epoch {epoch + 1:>2}: train_loss={train_loss:.4f} val_loss={val_loss:.4f} "
            f"val_acc={val_acc * 100:.1f}% lr={lr_now:.6f}{marker}"
        )

    if no_improve >= patience:
        print(f"  Early stopping at epoch {epoch + 1}")
        break

# Restore best model
dnn.load_state_dict(best_state)
print(f"\nRestored best model (val_loss={best_val_loss:.4f})")

In [ ]:
# Evaluate DNN
dnn.eval()
with torch.no_grad():
    y_prob_dnn = dnn(X_te_t).cpu().numpy().flatten()
y_pred_dnn = (y_prob_dnn >= 0.5).astype(int)
pnl_dnn = simulate_pnl(y_pred_dnn, y_test, asks_test)

ev_dnn = Evaluator(y_test, y_pred_dnn, y_prob_dnn, title="Deep Neural Network")
ev_dnn.full_report()
print(f"\nParameters: {n_params_dnn:,}")
print(
    f"PnL: ${pnl_dnn['pnl']:+.2f}  Bets: {pnl_dnn['n_bets']}  WR: {pnl_dnn['win_rate'] * 100:.1f}%  ROI: {pnl_dnn['roi']:+.1f}%"
)

all_results.append({"model": "DNN", "params": n_params_dnn, "ev": ev_dnn, "pnl": pnl_dnn})

## 7. Final Comparison

**What:** All four models on the same test set, same data, same seed. This is the definitive comparison.

**How to read:**
- **Accuracy** = binary prediction correctness
- **MSE/R²** = probability calibration quality
- **ROI** = real profitability after entry costs
- **Parameters** = model complexity (more params = higher overfitting risk)
- The best model has the **highest ROI** with an acceptable parameter count

In [ ]:
print(f"{'Model':<22} {'Params':>10} {'Accuracy':>9} {'MSE':>8} {'R²':>8} {'F1':>8} {'WR':>7} {'ROI':>8} {'PnL':>10}")
print("-" * 95)
for r in all_results:
    ev = r["ev"]
    p = r["pnl"]
    print(
        f"{r['model']:<22} {r['params']:>10,} {ev.accuracy * 100:>8.2f}% {ev.mse:>8.4f} {ev.r2 * 100:>7.1f}% "
        f"{ev.f1 * 100:>7.1f}% {p['win_rate'] * 100:>6.1f}% {p['roi']:>+7.1f}% ${p['pnl']:>+9.2f}"
    )
print(f"{'Random baseline':<22} {'—':>10} {'50.00%':>9}")

In [ ]:
# Visual comparison
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
model_names = [r["model"] for r in all_results]

# Accuracy
accs = [r["ev"].accuracy * 100 for r in all_results]
axes[0].barh(model_names, accs, color="steelblue", edgecolor="white")
axes[0].axvline(50, color="red", linestyle="--", alpha=0.3)
axes[0].set_xlabel("Accuracy (%)")
axes[0].set_title("Accuracy")
for i, v in enumerate(accs):
    axes[0].text(v + 0.3, i, f"{v:.1f}%", va="center", fontsize=9)

# ROI
rois = [r["pnl"]["roi"] for r in all_results]
colors = ["#2ecc71" if v > 0 else "#e74c3c" for v in rois]
axes[1].barh(model_names, rois, color=colors, edgecolor="white")
axes[1].axvline(0, color="gray", linestyle="-", linewidth=0.5)
axes[1].set_xlabel("ROI (%)")
axes[1].set_title("Return on Investment")
for i, v in enumerate(rois):
    axes[1].text(v + 0.1 if v >= 0 else v - 1.5, i, f"{v:+.1f}%", va="center", fontsize=9)

# Parameters (log scale)
params = [r["params"] for r in all_results]
axes[2].barh(model_names, params, color="mediumpurple", edgecolor="white")
axes[2].set_xscale("log")
axes[2].set_xlabel("Parameters (log scale)")
axes[2].set_title("Model Complexity")
for i, v in enumerate(params):
    axes[2].text(v * 1.2, i, f"{v:,}", va="center", fontsize=9)

plt.tight_layout()
plt.show()

## 8. Conclusion — Honest Results with Candle-Level Split

### Results (candle-level split, e>=30%, bid<=0.85)

| Model | Parameters | Accuracy | R² | Win Rate | ROI | PnL |
|-------|-----------|----------|-----|----------|-----|-----|
| **LogisticRegression** | **55** | **61.34%** | **3.8%** | **60.8%** | **-6.6%** | **-$158** |
| RandomForest | 329,564 | 62.25% | 7.2% | 61.8% | -6.5% | -$158 |
| XGBoost | 23,238 | 57.28% | -26.5% | 57.5% | -8.1% | -$168 |
| DNN (20 epochs) | 141,569 | 52.97% | — | 52.8% | -12.7% | -$242 |
| Random | — | 50.0% | 0.0% | — | — | — |

### Previous Results Were Fake

With the snapshot-level random split, RF showed 98%, XGBoost 100%, and DNN 99.7% accuracy. **With candle-level splitting, the truth emerges:**

- **RandomForest:** 98% → **62%**. Lost 36 points. Was memorizing candle identity.
- **XGBoost:** 100% → **57%**. Lost 43 points. Gradient boosting was perfectly overfitting to candle leakage.
- **DNN:** 99.7% → **53%**. Lost 47 points. The most parameters, the most overfitting. Nearly random on unseen candles.
- **LogisticRegression:** 70% → **61%**. Lost only 9 points. Too simple to memorize, so it was the most honest all along.

### Parameter Count vs Honest Accuracy

| Model | Parameters | Accuracy | Accuracy per 1K params |
|-------|-----------|----------|----------------------|
| LogisticRegression | 55 | 61.34% | 1,115%/K |
| XGBoost | 23,238 | 57.28% | 2.46%/K |
| DNN | 141,569 | 52.97% | 0.37%/K |
| RandomForest | 329,564 | 62.25% | 0.19%/K |

LogisticRegression is **5,700x more parameter-efficient** than RandomForest and achieves essentially the same accuracy. The complex models are spending their capacity on noise.

### Why Complex Models Failed

1. **Too few candles** — 654 training candles is tiny. RF needs thousands of diverse examples to generalize. XGBoost and DNN need even more.
2. **High within-candle correlation** — 100 snapshots per candle means 100 near-identical rows. With snapshot-level split, models learned "this BTC price = this candle = this outcome." With candle-level split, that shortcut is gone.
3. **XGBoost overfit hardest** — gradient boosting is designed to reduce training error aggressively. With leakage, it drove error to zero. Without it, the sequential tree corrections hurt generalization.
4. **DNN lacks data** — 141K parameters on 30K training samples (from 654 candles) is massive overfitting territory. Rule of thumb: need 10x more samples than parameters.

### The Real Answer

**No model is profitable yet with candle-level splitting.** All models achieve 53-62% accuracy on unseen candles, but the bid-ask spread on Polymarket tokens makes this unprofitable. The edge exists (all beat 50%) but it's not enough.

### What Would Make It Profitable

1. **5-10x more candles** — 5,000+ candles would give RF and XGBoost enough data to potentially find non-linear patterns that LogisticRegression can't.
2. **Better features** — current features are mostly derived from within the same candle. Cross-candle features (market regime, volatility clusters, time-of-day patterns) might carry signal that generalizes.
3. **Lower entry costs** — if Polymarket spreads narrow, the same 62% accuracy becomes profitable.
4. **Confidence filtering** — only betting on high-confidence predictions could push the effective win rate above the profitability threshold.